# Chapter 3 - Lab 1: <font color='blue'>LangChain Agent</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using LangChain** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

You will see how LangChain's `create_agent` API wires together:

* Two **tools** (one for fetching stock data, one for computing the P/E ratio),
* A **system prompt** that frames the model as a finance analyst,
* An **LLM** (`gpt-4o-mini`) that decides which tool to call and when.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **LangChain** (`langchain`) — agent abstraction with `create_agent` and tool decorators.
* **OpenAI** `gpt-4o-mini` — the underlying LLM driving tool calls.
* **Pydantic** — typed input schemas for the tools.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [ ]:
%pip install -q langchain langchain-openai pydantic python-dotenv

## 2. Set up the API key(s)

If you are running this notebook in **Google Colab**, add your OpenAI key in the left vertical menu (the *key* icon) under the name `OPENAI_API_KEY`.

If you are running locally, replace the cell below with `os.environ['OPENAI_API_KEY'] = '...'` or load it from a `.env` file.

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

## 4. Create the LangChain agent

LangChain's `create_agent` accepts a list of tools, a model name (using the `provider:model` string convention), and a system prompt. The framework handles tool registration, schema inference from type hints, and the agent loop under the hood.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    tools=[get_stock_data, compute_pe],
    model="openai:gpt-4o-mini",
    system_prompt=system_message,
)

## 5. Run the agent

We invoke the agent with the user's request. LangChain returns the entire message history, including the tool calls the agent made and their results — useful for debugging and for understanding the model's reasoning.

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": input_message},
    ]
})

## 6. Inspect the agent's trace

We walk through the returned messages and print each tool call and each AI response. This trace is the single most useful artifact when comparing frameworks — it shows *how* the agent reasoned, not just the final answer.

In [ ]:
for msg in result.get("messages", []):
    msg_type = type(msg).__name__
    if msg_type == "ToolMessage":
        print(f"[Tool: {msg.name}]\n{msg.content}\n")
    elif msg_type == "AIMessage" and msg.content:
        print(f"[Agent Response]\n{msg.content}\n")

## 7. Results

You should see the agent call `get_stock_data` twice (once per ticker), then call `compute_pe` twice to compute each P/E ratio, and finally produce a short memo comparing the two.

**What to notice about LangChain specifically:**

* The `provider:model` string convention is elegant — switching to another provider is a one-line change.
* `create_agent` infers tool schemas automatically from function signatures and docstrings, which keeps the lab short.
* The returned `messages` list contains typed objects (`AIMessage`, `ToolMessage`, …), making post-hoc analysis straightforward.